# GRPO — checkpointed group-relative policy optimization (run on the A100)

Fresh Colab, **A100 + High-RAM strongly recommended**, then run top to bottom. This trains the P6 QLoRA policy with GRPO to directly optimize the **free-running behavioral reward**, so free-running **greedy** behavioral fidelity climbs from ~0.159 toward/past the oracle ~0.42 — without a reranker at deploy. Loop = docs/grpo/ (`sft.grpo_train`).

**Anytime / resumable:** `latest/` is written every `ckpt_every` steps and on SIGINT/SIGTERM; a guard-vetoed `best/` is a separate, deployable dir. Interrupt any time and re-run the run cell with the same `--run-id` to resume. **Submit `best/`, never `latest/`.**

> ⚠ **Prerequisite:** this notebook clones the repo and checks out **`feat/grpo`** — that branch (the GRPO code) must be committed + pushed to `origin` first. Change `BRANCH` below if you merged it elsewhere.


In [ ]:
# CELL 1 — provision the runtime (clone, install, HF token, stage the corpus)  [from phase1 CELL 1]
import os, pathlib, subprocess, sys
REPO = '/content/SLM'
BRANCH = 'feat/grpo'                      # <-- the GRPO branch; must be pushed to origin
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
if not os.environ.get('SLM_PROVISIONED'):
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
import getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = _env_token('HF_TOKEN') or getpass.getpass('HF_TOKEN (read, for staging + adapter): ').strip()
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'          # lowercase corpus (the Colab case trap)
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')

In [ ]:
# CELL 2 — build base_resized + download the P6 two-stage adapter (policy init AND frozen ref)  [from phase1 CELL 2]
import os, pathlib, subprocess, sys, json
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
D = pathlib.Path('models/base_resized')
if not ((D / 'vocab_resize_manifest.json').is_file() and (D / 'preprocessor_config.json').is_file()):
    print('building base_resized (loads base in fp32 on CPU ~12GB; A100/High-RAM recommended) ...')
    subprocess.run([sys.executable, '-m', 'sft.vocab_resize', '--out', 'models/base_resized'],
                   check=True, env={**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'})
vr = json.load(open(D / 'vocab_resize_manifest.json'))
print('base_resized OK; identity:', vr['tokenizer_version'])

from huggingface_hub import snapshot_download
ADAPTER_SUBFOLDER = 'p6_twostage_d0f9c744_smokefull'
snapshot_download(repo_id='ericrcwu/LUT_SLM_sft_adapters', allow_patterns=[ADAPTER_SUBFOLDER + '/*'],
                  local_dir='models/sft_adapters', token=os.environ.get('HF_TOKEN'))
INIT_ADAPTER = 'models/sft_adapters/' + ADAPTER_SUBFOLDER
am = json.load(open(pathlib.Path(INIT_ADAPTER) / 'adapter_manifest.json'))
assert am['tokenizer_version'] == vr['tokenizer_version'], 'adapter/tokenizer identity mismatch'
assert pathlib.Path(INIT_ADAPTER, 'adapter_model.safetensors').is_file(), INIT_ADAPTER
print('P6 init adapter ready:', INIT_ADAPTER, '| step:', am.get('adapter_step'))

In [ ]:
# CELL 3 — GPU + stack sanity (must be an A100/Ampere for bf16; fp16 fallback on T4)
import torch, transformers, bitsandbytes as bnb, peft, accelerate, qwen_vl_utils
from transformers import Qwen2_5_VLForConditionalGeneration          # the VL class must exist
assert torch.cuda.is_available(), 'no CUDA — wrong runtime (need an A100)'
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| peft', peft.__version__, '| gpu', torch.cuda.get_device_name(0))
# off-GPU unit tests (fast) — reward/rollout/loss/harness parity before burning GPU time
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q',
                    'tests/test_grpo_reward.py', 'tests/test_rollout.py',
                    'tests/test_grpo_loss.py', 'tests/test_grpo_train.py'],
                   capture_output=True, text=True)
print(r.stdout[-1500:]); print('unit tests', 'PASS' if r.returncode == 0 else 'FAIL')

## SMOKE — 4 steps, tiny rounds, frequent ckpt/eval (acceptance B)

Expect: rollouts produce ≥1 gradable sample (else `[grpo][ABORT]`), finite loss, `latest/` + `best/` + `eval_log.jsonl` appear, and a `{"grpo_summary": ...}` line. `--eval-every 2 --ckpt-every 2` exercise the periodic-eval / BEST / atomic-`latest` paths inside the tiny run.


In [ ]:
# CELL 4 — SMOKE run
import os, sys, subprocess, json, pathlib
RUN_ID = 'grpo_smoke'
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.grpo_train',
       '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
       '--run-id', RUN_ID, '--total-steps', '4', '--prompts-per-round', '2',
       '--eval-every', '2', '--ckpt-every', '2', '--eval-limit', '16']
print('running:', ' '.join(cmd), '\n')
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-6000:])
if p.returncode != 0:
    print('--- STDERR tail ---\n', p.stderr[-3000:])
run = pathlib.Path('models/sft_adapters', RUN_ID)
print('\nlatest/ exists:', (run / 'latest').is_dir(), '| best/ exists:', (run / 'best').is_dir(),
      '| eval_log.jsonl:', (run / 'eval_log.jsonl').is_file())
assert p.returncode == 0 and (run / 'latest').is_dir(), 'smoke failed — see stderr above'

## RESUME check — re-run the SAME run-id with a higher step budget (acceptance B, resume)

Proves the anytime/resume path: it reloads `latest/`, prints a `[grpo] RESUME step=4 ...` line, and continues to step 8 with `best_summary` preserved. A real SIGINT/Colab-preemption mid-run lands in the same `latest/` and resumes identically — just re-run the run cell.


In [ ]:
# CELL 5 — RESUME the smoke (same --run-id, higher --total-steps)
import os, sys, subprocess, pathlib
RUN_ID = 'grpo_smoke'
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.grpo_train',
       '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
       '--run-id', RUN_ID, '--total-steps', '8', '--prompts-per-round', '2',
       '--eval-every', '2', '--ckpt-every', '2', '--eval-limit', '16']
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-6000:])
assert 'RESUME' in p.stdout, 'expected a [grpo] RESUME line (resume from latest/ did not trigger)'
print('\nRESUME OK — step continued from the smoke checkpoint.')

## FULL RUN — the real GRPO run (acceptance C)

Uses `configs/candidate_grpo.json` defaults (`total_steps=500`, `group_size=8`, `eval_every=20`, `ckpt_every=20`). Long-running but **anytime**: if the VM preempts, just re-run this cell (same `--run-id`) to resume from `latest/`. Watch `models/sft_adapters/grpo_r1/eval_log.jsonl` for the greedy fidelity trend + guard panel.


In [ ]:
# CELL 6 — FULL RUN (resumable; re-run this exact cell after any preemption)
import os, sys, subprocess, pathlib, json
RUN_ID = 'grpo_r1'
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.grpo_train',
       '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
       '--run-id', RUN_ID]      # total_steps/group_size/eval_every/ckpt_every from the config
print('running:', ' '.join(cmd), '\n(this is long; tail eval_log.jsonl in another cell to watch)\n')
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-8000:])
if p.returncode != 0:
    print('--- STDERR tail ---\n', p.stderr[-3000:])
for line in p.stdout.splitlines():
    if line.strip().startswith('{') and 'grpo_summary' in line:
        print('\nSUMMARY:', json.loads(line)['grpo_summary'])

In [ ]:
# CELL 6b — (optional) watch the eval log while CELL 6 runs (run in a separate cell)
import json, pathlib
run = pathlib.Path('models/sft_adapters', 'grpo_r1')
log = run / 'eval_log.jsonl'
if log.is_file():
    for line in log.read_text().splitlines():
        r = json.loads(line)
        print('step %4s | greedy_fid=%s collapse=%s degen=%s entropy=%s dE=%s kl=%s reward=%s %s' % (
            r.get('step'), r.get('behavioral_fidelity_mean'), r.get('collapse_rate'),
            r.get('degenerate_rate'), r.get('code_entropy_norm_mean'), r.get('decoded_delta_e_mean'),
            r.get('kl_to_ref'), r.get('reward_mean'), '<-- BEST' if r.get('is_best') else ''))
else:
    print('no eval_log yet')

## GATE — score `best/` on the full holdout (the real acceptance)

**Pass = free-running GREEDY `behavioral_fidelity_mean` beats 0.159 and moves toward/past oracle ~0.42**, with every anti-hacking guard healthy (collapse/degenerate/ΔE not rising, entropy not collapsing) and `oracle@32` not regressing. The teacher-forced `METRIC=` line is blind to collapse — **not** the gate.


In [ ]:
# CELL 7 — GATE: free-running greedy (+sample) behavioral fidelity on best/
import os, sys, subprocess, json
BEST = 'models/sft_adapters/grpo_r1/best'
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
cmd = [sys.executable, '-m', 'sft.score_tokens',
       '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
       '--adapter', BEST, '--behavioral-sampling', 'both', '--behavioral-limit', '0']
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-6000:])
summary = None
for line in p.stdout.splitlines():
    if line.strip().startswith('{') and 'score_summary' in line:
        summary = json.loads(line)['score_summary']
if summary:
    g = summary.get('behavioral') or {}
    s = summary.get('behavioral_sampled') or {}
    print('\n================= GATE =================')
    print('GREEDY  behavioral_fidelity_mean =', g.get('behavioral_fidelity_mean'),
          '(baseline greedy 0.159; best-of-N / oracle ~0.42)')
    print('  greedy collapse_rate=%s degenerate=%s deltaE=%s entropy=%s' % (
        g.get('collapse_rate'), g.get('degenerate_rate'),
        g.get('decoded_delta_e_mean'), g.get('code_entropy_norm_mean')))
    print('SAMPLE  behavioral_fidelity_mean =', s.get('behavioral_fidelity_mean'))

In [ ]:
# CELL 8 — COVERAGE: oracle@N on best/ (must NOT regress) + best-of-N head-to-head
import os, sys, subprocess
BEST = 'models/sft_adapters/grpo_r1/best'
env = {**os.environ, 'SLM_ARTIFACT_ROOT': '/content/slm'}
subprocess.run([sys.executable, '-m', 'eval.oracle_at_n',
                '--config', 'configs/candidate_grpo.json', '--resized-model', 'models/base_resized',
                '--adapter', BEST, '--limit', '0', '--n', '32', '--chunk', '16',
                '--temperatures', '0.7,1.0'], env=env)
subprocess.run([sys.executable, '-m', 'eval.best_of_n',
                '--config', 'configs/candidate_grpo.json', '--adapter', BEST,
                '--limit', '0', '--n', '16', '--temperature', '1.0'], env=env)
print('\nRead: oracle@32 >= init ~0.42 (no coverage destroyed) AND greedy (CELL 7) moved up toward best_of_N.')

In [ ]:
# CELL 9 — (optional) upload best/ to the HF adapters repo (needs a WRITE token)
import os, getpass
from huggingface_hub import upload_folder
if not os.environ.get('HF_WRITE_TOKEN'):
    os.environ['HF_WRITE_TOKEN'] = getpass.getpass('HF_WRITE_TOKEN (SLM_Alpha_Write): ').strip()
RUN_ID = 'grpo_r1'
upload_folder(repo_id='ericrcwu/LUT_SLM_sft_adapters', folder_path=f'models/sft_adapters/{RUN_ID}/best',
              path_in_repo=f'{RUN_ID}_best', token=os.environ['HF_WRITE_TOKEN'])
print('uploaded best/ ->', f'ericrcwu/LUT_SLM_sft_adapters/{RUN_ID}_best')